# XGBOOST SINGLE

# Overview
This repository provides code implementation for training Gradient Boosting Models (GBMs), a popular machine learning technique for both classification and regression tasks. GBMs are ensemble methods that combine the predictions of several base estimators to improve accuracy and generalization performance.



# Inference
[[MITSUI-CPC] Gradient Boosting Models (Inference)](https://www.kaggle.com/code/takaito/mitsui-cpc-gradient-boosting-models-inference)

# Tips
## 1. CV Strategy
By setting kfold = KFold(n_splits=CFG.N_SPLIT, shuffle=False), the data is being loaded in chronological order, so the splitting is performed based on the time series.

## 2. feature importance
In LightGBM, we save the feature importance. This allows you to check which features are effective and can provide insights for removing unnecessary features or creating new ones, so please make use of it.

To be updated!! (I plan to add more hints if the number of votes increases.)

In [1]:
# ====================================================
# Library
# ====================================================
import os
import gc
import warnings
warnings.filterwarnings('ignore')
import random
import scipy as sp
import numpy as np
import pandas as pd
import polars as pl
from glob import glob
from pathlib import Path
import joblib
import pickle
import itertools
from tqdm.auto import tqdm

import torch
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split, GroupKFold
from sklearn.metrics import log_loss, roc_auc_score, matthews_corrcoef, f1_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import Pool, CatBoostRegressor, CatBoostClassifier

In [2]:
!mkdir oof
!mkdir models

mkdir: oof: File exists
mkdir: models: File exists


In [ ]:
# ====================================================
# Configurations
# ====================================================
class CFG:
    VER = 1
    AUTHOR = 'takaito'
    COMPETITION = 'mitsui-commodity-prediction-challenge'
    DATA_PATH = Path('./mitsui-commodity-prediction-challenge')
    OOF_DATA_PATH = Path('./oof')
    MODEL_DATA_PATH = Path('./models')
    METHOD_LIST = ['xgboost']
    USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SPLIT = 6
    metric = 'rmse'
    metric_maximize_flag = False

    num_boost_round = 5000
    early_stopping_round = 200
    verbose = 50
    
    regression_xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': 0.02, 
        'max_depth': 10,
        'min_child_weight': 12,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'reg_lambda': 3.0,
        'reg_alpha': 0.1,
        'gamma': 0.1,
        'max_leaves': 64,
        'tree_method': 'gpu_hist' if USE_GPU else 'hist',
        'enable_categorical': True,
        'max_cat_to_onehot': 1,
        'random_state': SEED,
    }
    
    # diff params for each fold

    PREFIX = f'{AUTHOR}_seed{SEED}_ver{VER}'

In [4]:
# ====================================================
# Seed everything
# ====================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
seed_everything(CFG.SEED)

In [5]:
SOLUTION_NULL_FILLER = -999999

def rank_correlation_sharpe_ratio(merged_df: pd.DataFrame) -> float:
    """
    Calculates the rank correlation between predictions and target values,
    and returns its Sharpe ratio (mean / standard deviation).

    :param merged_df: DataFrame containing prediction columns (starting with 'prediction_')
                      and target columns (starting with 'target_')
    :return: Sharpe ratio of the rank correlation
    :raises ZeroDivisionError: If the standard deviation is zero
    """
    prediction_cols = [col for col in merged_df.columns if col.startswith('prediction_')]
    target_cols = [col for col in merged_df.columns if col.startswith('target_')]

    def _compute_rank_correlation(row):
        non_null_targets = [col for col in target_cols if not pd.isnull(row[col])]
        matching_predictions = [col for col in prediction_cols if col.replace('prediction', 'target') in non_null_targets]
        if not non_null_targets:
            raise ValueError('No non-null target values found')
        if row[non_null_targets].std(ddof=0) == 0 or row[matching_predictions].std(ddof=0) == 0:
            raise ZeroDivisionError('Denominator is zero, unable to compute rank correlation.')
        return np.corrcoef(row[matching_predictions].rank(method='average'), row[non_null_targets].rank(method='average'))[0, 1]

    daily_rank_corrs = merged_df.apply(_compute_rank_correlation, axis=1)
    std_dev = daily_rank_corrs.std(ddof=0)
    if std_dev == 0:
        raise ZeroDivisionError('Denominator is zero, unable to compute Sharpe ratio.')
    sharpe_ratio = daily_rank_corrs.mean() / std_dev
    return float(sharpe_ratio)


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculates the rank correlation between predictions and target values,
    and returns its Sharpe ratio (mean / standard deviation).
    """
    del solution[row_id_column_name]
    del submission[row_id_column_name]
    assert all(solution.columns == submission.columns)

    submission = submission.rename(columns={col: col.replace('target_', 'prediction_') for col in submission.columns})

    # Not all securities trade on all dates, but solution files cannot contain nulls.
    # The filler value allows us to handle trading halts, holidays, & delistings.
    solution = solution.replace(SOLUTION_NULL_FILLER, None)
    return rank_correlation_sharpe_ratio(pd.concat([solution, submission], axis='columns'))

In [6]:
def build_solution_and_submission(
    fold_date_ids,      # 当前fold有效的时间id集合
    train_labels_wide,  # 原始训练标签 (宽表形式)
    valid_df_long,      # 验证集 (长表形式)
    valid_pred,         # 验证集预测结果
    time_key,           # 时间列关键字
    n_targets=None,
    SOLUTION_NULL_FILLER=-999999,
):
    """
    构造 solution_df 和 submission_df
    """

    # 1) 取该折的“完整标签矩阵” (sol_base)
    target_cols = [c for c in train_labels_wide.columns if c.startswith("target_")]
    if n_targets is not None:
        target_cols = [f"target_{i}" for i in range(n_targets)]

    # 只保留当前折的日期
    sol_base = train_labels_wide[train_labels_wide[time_key].isin(fold_date_ids)].copy()

    # 确保列齐全有序
    sol_base = sol_base[[time_key] + target_cols]

    # 2) 构建 submission 的空白布局 (与 solution 行列完全一致)
    sub_base = sol_base[[time_key]].copy()
    for c in target_cols:
        sub_base[c] = np.nan

    # 3) 构造成宽表预测 (避免找列/缺行; 后面reindex补齐)
    vd = valid_df_long[[time_key, 'target_id']].copy()
    vd["pred"] = valid_pred
    vd["target_col"] = "target_" + vd["target_id"].astype(int).astype(str)

    pred_wide = vd.pivot_table(
        index=time_key, columns="target_col", values="pred", aggfunc="first"
    )

    # 对齐到完整矩阵
    pred_wide = pred_wide.reindex(
        index=sol_base[time_key].values, columns=target_cols
    )

    # 4) 对于原本缺失的 {date, target}, submission 必须保持 NaN
    submission = sub_base.copy()

    # 找到可以覆盖的位置 (目标非缺失)
    valid_mask = ~sol_base[target_cols].isna()

    # 填写预测
    submission.loc[:, target_cols] = np.where(
        valid_mask, pred_wide.values, np.nan
    )

    # 5) 输出官方格式 (row_id 放到第一列)
    solution = sol_base.copy()
    solution[target_cols] = solution[target_cols].fillna(SOLUTION_NULL_FILLER)
    solution = solution.rename(columns={time_key: "row_id"})

    submission = submission.rename(columns={time_key: "row_id"})

    return solution, submission


def score_official_compatible(solution_df: pd.DataFrame, submission_df: pd.DataFrame, row_id_column_name: str = "row_id",) -> float:
    """
    Compatible scoring function using official-like evaluation.
    """

    # 直接用提供的官方代码: 把 -999999 换为缺失，并做 Sharpe-like
    sol = solution_df.copy()
    sub = submission_df.copy()

    del sol[row_id_column_name]
    del sub[row_id_column_name]

    assert list(sol.columns) == list(sub.columns)

    sub = sub.rename(
        columns={c: c.replace("target_", "prediction_") for c in sub.columns}
    )
    sol = sol.replace(SOLUTION_NULL_FILLER, None)

    # 官方 rank_correlation_sharpe_ratio(merged) 会在遇到方差为0时报错
    prediction_cols = [c for c in sub.columns if c.startswith("prediction_")]

    merged = pd.concat([sol, sub], axis=1)

    # 调用官方定义的 rank_correlation_sharpe_ratio
    return rank_correlation_sharpe_ratio(merged)

In [7]:

def xgboost_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    CFG=CFG,
):
    xgb_train = xgb.DMatrix(data=x_train, label=y_train, enable_categorical=True)
    xgb_valid = xgb.DMatrix(data=x_valid, label=y_valid, enable_categorical=True)
    model = xgb.train(
        CFG.regression_xgb_params,
        dtrain=xgb_train,
        num_boost_round=CFG.num_boost_round,
        evals=[(xgb_train, "train"), (xgb_valid, "eval")],
        early_stopping_rounds=CFG.early_stopping_round,
        verbose_eval=CFG.verbose,
    )

    # Predict validation
    valid_pred = model.predict(xgb_valid)

    return model, valid_pred

def gradient_boosting_model_cv_training(method: str, train_df: pd.DataFrame, features: list, target_cols: list):
    # Create a numpy array to store out of folds predictions
    oof_predictions = np.zeros(len(train_df))
    oof_fold = np.zeros(len(train_df))
    scores = []
    
    for fold in range(CFG.N_SPLIT):
        print('-'*50)
        print(f'{method} training fold {fold+1}')
        
        # x_train = train_df[train_df['cv_flag']!=fold+1][features]
        # y_train = train_df[train_df['cv_flag']!=fold+1]['target']
        # valid_df = train_df[train_df['cv_flag']==fold+1].copy()
        # x_valid = valid_df[features]
        # y_valid = valid_df['target']
        
        # 训练数据和验证数据必须得满足时序特性 so <
        # 基于过去的数据预测未来的数据
        x_train = train_df[train_df['fold'] < fold][features]
        y_train = train_df[train_df['fold'] < fold]['target']

        valid_df = train_df[train_df['fold'] == fold].copy()
        x_valid = valid_df[features]
        y_valid = valid_df['target']
        
        
        
        # if method == 'lightgbm':
        #     model, valid_pred = lightgbm_training(x_train, y_train, x_valid, y_valid)
        #     ## 2. feature importance
        #     importance_df = pd.DataFrame(model.feature_importance(), index=features, columns=['importance']).reset_index()
        #     importance_df.to_csv(CFG.MODEL_DATA_PATH / f'{method}_fold{fold + 1}_{CFG.PREFIX}_importance.csv', index=False)
        if method == 'xgboost':
            model, valid_pred = xgboost_training(x_train, y_train, x_valid, y_valid)
        # if method == 'catboost':
        #     model, valid_pred = catboost_training(x_train, y_train, x_valid, y_valid)
            
        # calculate cv score
        # use official 
        fold_date_ids = train_df.loc[train_df['fold']==fold, 'date_id'].unique()
        solution_df, submission_df = build_solution_and_submission(
            fold_date_ids=fold_date_ids,
            train_labels_wide=raw_label_df,
            valid_df_long=valid_df,
            valid_pred=valid_pred,
            time_key='date_id',
            n_targets=424
        )

        fold_score = score_official_compatible(solution_df, submission_df, row_id_column_name='row_id')
        print(f'Fold {fold+1} score: {fold_score:.6f}')
        scores.append(fold_score)

        # Save best model
        pickle.dump(model, open(CFG.MODEL_DATA_PATH / f'{method}_fold{fold + 1}_{CFG.PREFIX}.pkl', 'wb'))
        # Add to out of folds array
        # oof_predictions[train_df['cv_flag']==fold+1] = valid_pred
        
        oof_predictions[train_df['fold']==fold] = valid_pred
        
        del x_train, x_valid, y_train, y_valid, model, valid_pred, valid_df
        gc.collect()


    print(f"Final score: {sum(scores)/len(scores):.6f}")
    train_df['pred'] = oof_predictions
    # Create a dataframe to store out of folds predictions
    np.save(CFG.OOF_DATA_PATH / f'oof_{method}_{CFG.PREFIX}', oof_predictions)
    

In [8]:
train_df = pl.read_csv(CFG.DATA_PATH / f'train.csv').to_pandas()
train_labels_df = pl.read_csv(CFG.DATA_PATH / f'train_labels.csv').to_pandas()


In [9]:
raw_label_df = train_labels_df.copy()

In [10]:
original_features = list(train_df.columns[1:])

In [11]:
target_cols = list(train_labels_df.columns[1:])

In [12]:
# train_df['cv_flag'] = pd.qcut(train_df.index, CFG.N_SPLIT, labels=False) + 1
train_df["fold"] = -1
train_df["fold"].iloc[1827-90*5:1827-90*4] = 0
train_df["fold"].iloc[1827-90*4:1827-90*3] = 1
train_df["fold"].iloc[1827-90*3:1827-90*2] = 2
train_df["fold"].iloc[1827-90*2:1827-90] = 3
train_df["fold"].iloc[1827-90:1827] = 4
train_df["fold"].iloc[1827:] = 5   # = Public LB

In [13]:
def add_time_features(
                df: pd.DataFrame,
                date_col: str = "date_id",
                base_cols: list[str] | None = None,
                lags: list[int] = [1, 2, 3, 5],
                roll_windows: list[int] = [3, 5, 10],
                add_diff: bool = True,
                fillna_value: float | None = None) -> tuple[pd.DataFrame, list[str]]:
        """
        在df上按date_id排序, 给base_cols 增加：
                - 滞后特征 (col_lag{k})
                - 滚动均值/标准差 (col_rmean{w}, col_rstd{w})
                - 一阶差分 (col_diff1)
        返回: (df, 新增排名列表)
        """

        if base_cols is None: 
                base_cols = [c for c in df.columns if c not in ["date_id", "is_scored", "fold"]]

        # 按时间排序以避免数据穿越
        df = df.sort_values(date_col).copy()
        new_cols = []

        # 滞后特征
        for c in base_cols:
                s = df[c]
                for k in lags:
                        coln = f"{c}_lag{k}"
                        df[coln] = s.shift(k)
                        new_cols.append(coln)

        # # 流动统计，先shift（1）再rolling，确保只用过去的
        # for c in base_cols:
        #         s = df[c].shift(1)
        #         for w in roll_windows:
        #                 col_mean = f"{c}_rmean{w}"
        #                 col_std = f"{c}_rstd{w}"
        #                 # 使用 shift(1) 避免未来信息泄漏
        #                 df[col_mean] = s.rolling(w, min_periods=2).mean()
        #                 df[col_std] = s.rolling(w, min_periods=2).std(ddof=0)
        #                 new_cols.extend([col_mean, col_std])

        # # 一阶差分
        # if add_diff:
        #         for c in base_cols:
        #                 col_diff = f"{c}_diff1"
        #                 df[col_diff] = df[c] - df[f"{c}_lag1"]
        #                 new_cols.append(col_diff)


        # 处理缺失：XGB 可以 NaN; fill 成与推理一致的值
        if fillna_value is not None:
                df[new_cols] = df[new_cols].fillna(fillna_value)
                

        # 恢复原始顺序
        df = df.sort_index()

        return df, new_cols


In [14]:
# 0) 生成时序特征
base_for_timefe = [c for c in original_features]
train_df_time, time_cols = add_time_features(
        df=train_df,
        date_col='date_id',
        base_cols=base_for_timefe,
        lags=[1],
        # 可选: roll_windows, add_diff
)

original_features_plus = original_features + time_cols

# 1) 标签宽表 → 长表
labels_long = (
        train_labels_df
                .replace([np.inf, -np.inf], np.nan)
                .set_index('date_id')
                .stack(dropna=True)  # -> index: (date_id, target_col)
                .rename("target")
                .reset_index()
                .rename(columns={"level_1": "target_col"}) # ['date_id', 'target_col', 'target']
)
labels_long = labels_long[labels_long["target"].abs() <= 1e10]

# 2) 提取 target_id (int16) 并去掉 target_col, （int16/category）
labels_long["target_id"] = labels_long["target_col"].str.split("_").str[1].astype("int16")
labels_long = labels_long.drop(columns=["target_col"])

# ========= 3) 合并特征 =========
need_feat_cols = ['date_id', 'fold'] + original_features_plus
# 降为度，进一步节省内存
for c in original_features_plus:
        if pd.api.types.is_float_dtype(train_df_time[c]):
                train_df_time[c] = train_df_time[c].astype("float32")

training_df = labels_long.merge(train_df_time[need_feat_cols], on='date_id', how="left", copy=False)

# ========= 4) 类型转换 (节省内存 & 保证训练稳定) =========
training_df["target"] = training_df["target"].astype("float32")
training_df["target_id"] = training_df["target_id"].astype("category")

In [15]:
# training_df = []
# for j, target_col in enumerate(target_cols):
#     temp_train_df = train_df.copy()
#     temp_train_df.rename(columns={'row_id': 'original_row_id'}, inplace=True)  # Add this line
#     temp_train_df['target_id'] = j
#     y = train_labels_df[target_col].values
#     temp_train_df['target'] = y
#     mask = ~(np.isnan(y) | np.isinf(y) | (np.abs(y) > 1e10))
#     training_df.append(temp_train_df[mask].copy())
    
# # training_df = pd.concat(training_df).reset_index(drop=True)
# training_df = pd.concat(training_df, ignore_index=True)
# training_df.rename(columns={'row_id': 'original_row_id'},  inplace=True)
# training_df['target_id'] = training_df['target_id'].astype('category')
# training_df = training_df.reset_index().rename(columns={'index': 'row_id'})

In [16]:
for method in CFG.METHOD_LIST:
    gradient_boosting_model_cv_training(method, training_df.copy(), original_features + ['target_id'], target_cols)

--------------------------------------------------
xgboost training fold 1
[0]	train-rmse:0.03367	eval-rmse:0.02431
[50]	train-rmse:0.03311	eval-rmse:0.02431
[100]	train-rmse:0.03296	eval-rmse:0.02431
[150]	train-rmse:0.03293	eval-rmse:0.02431
[200]	train-rmse:0.03290	eval-rmse:0.02432
[250]	train-rmse:0.03288	eval-rmse:0.02432
[300]	train-rmse:0.03288	eval-rmse:0.02432
[341]	train-rmse:0.03284	eval-rmse:0.02432
Fold 1 score: 0.095982
--------------------------------------------------
xgboost training fold 2
[0]	train-rmse:0.03317	eval-rmse:0.02461
[50]	train-rmse:0.03242	eval-rmse:0.02458
[100]	train-rmse:0.03223	eval-rmse:0.02457
[150]	train-rmse:0.03218	eval-rmse:0.02457
[200]	train-rmse:0.03216	eval-rmse:0.02456
[250]	train-rmse:0.03215	eval-rmse:0.02456
[300]	train-rmse:0.03214	eval-rmse:0.02456
[350]	train-rmse:0.03212	eval-rmse:0.02456
[400]	train-rmse:0.03212	eval-rmse:0.02456
[450]	train-rmse:0.03212	eval-rmse:0.02456
[500]	train-rmse:0.03212	eval-rmse:0.02456
[536]	train-rmse